In [2]:
!pip install -q roboflow pyyaml

In [3]:
from google.colab import drive, userdata
from roboflow import Roboflow
from pathlib import Path

drive.mount('/content/drive')

ROBOFLOW_KEY = userdata.get("ROBFLOW_KEY")

rf = Roboflow(api_key=ROBOFLOW_KEY)

BASE_DIR = Path(
    "/content/drive/MyDrive/Hazard_Detection/"
    "raw_datasets/doors"
)

SOURCE_DIR = BASE_DIR / "source_datasets"

Mounted at /content/drive


In [4]:

# ==========================================
# DOWNLOAD DOOR DATASET - ROBOFLOW VERSION 3
# ==========================================

from pathlib import Path

DOWNLOAD_DIR = SOURCE_DIR / "door_v3"

if DOWNLOAD_DIR.exists():
    raise FileExistsError(
        f"Destination already exists: {DOWNLOAD_DIR}\n"
        "Please inspect it before downloading again."
    )

door_project = rf.workspace(
    "object-detect-zjtnj"
).project(
    "door-jqeug"
)

print("PROJECT:", door_project.id)

door_version = door_project.version(3)

print("VERSION:", door_version.version)

door_dataset = door_version.download(
    "yolov8",
    location=str(DOWNLOAD_DIR)
)

print("\nDOWNLOAD SUCCESSFUL!")
print("DATASET LOCATION:", door_dataset.location)


loading Roboflow workspace...
loading Roboflow project...
PROJECT: object-detect-zjtnj/door-jqeug
VERSION: 3



Extracting Dataset Version Zip to /content/drive/MyDrive/Hazard_Detection/raw_datasets/doors/source_datasets/door_v3 in yolov8:: 100%|██████████| 2012/2012 [00:23<00:00, 86.14it/s]


DOWNLOAD SUCCESSFUL!
DATASET LOCATION: /content/drive/MyDrive/Hazard_Detection/raw_datasets/doors/source_datasets/door_v3


In [5]:
# INSPECT DOOR DATASET

import yaml
from collections import Counter

DATASET_DIR = Path(door_dataset.location)

yaml_files = list(DATASET_DIR.rglob("data.yaml"))

if not yaml_files:
    raise FileNotFoundError("data.yaml not found!")

DATA_YAML = yaml_files[0]

with open(DATA_YAML, "r") as f:
    config = yaml.safe_load(f)

names = config["names"]

if isinstance(names, list):
    class_names = dict(enumerate(names))
else:
    class_names = {int(k): v for k, v in names.items()}

print("=" * 65)
print("ORIGINAL CLASS MAPPING")
print("=" * 65)

for class_id, name in class_names.items():
    print(f"{class_id}: {name}")

print("\nDATASET LOCATION:", DATASET_DIR)

total_instances = 0

for split in ["train", "valid", "test"]:

    LABEL_DIR = DATASET_DIR / "labels" / split
    IMAGE_DIR = DATASET_DIR / "images" / split

    if not LABEL_DIR.exists():
        print(f"\n{split.upper()}: NOT FOUND")
        continue

    counts = Counter()
    images_with_labels = 0

    for label_file in LABEL_DIR.glob("*.txt"):

        lines = [
            line.strip()
            for line in label_file.read_text().splitlines()
            if line.strip()
        ]

        if lines:
            images_with_labels += 1

        for line in lines:
            class_id = int(line.split()[0])
            counts[class_id] += 1

    instances = sum(counts.values())
    total_instances += instances

    image_count = (
        len(list(IMAGE_DIR.iterdir()))
        if IMAGE_DIR.exists() else 0
    )

    print("\n" + "=" * 65)
    print(split.upper())
    print("=" * 65)

    print("Images:", image_count)
    print("Images with labels:", images_with_labels)
    print("Total instances:", instances)

    for class_id, name in class_names.items():
        print(
            f"{class_id}: {name:<20} "
            f"{counts[class_id]}"
        )

print("\n" + "=" * 65)
print("TOTAL DOOR INSTANCES:", total_instances)
print("TARGET INSTANCES: 200")
print("=" * 65)

ORIGINAL CLASS MAPPING
0: door

DATASET LOCATION: /content/drive/MyDrive/Hazard_Detection/raw_datasets/doors/source_datasets/door_v3

TRAIN: NOT FOUND

VALID: NOT FOUND

TEST: NOT FOUND

TOTAL DOOR INSTANCES: 0
TARGET INSTANCES: 200


In [6]:

# ==========================================
# DIAGNOSE ROBOFLOW DOWNLOAD STRUCTURE
# ==========================================

print("DATASET DIR:", DATASET_DIR)
print("EXISTS:", DATASET_DIR.exists())

print("\n" + "=" * 65)
print("TOP-LEVEL CONTENTS")
print("=" * 65)

for item in sorted(DATASET_DIR.iterdir()):
    print(
        "[DIR] " if item.is_dir() else "[FILE]",
        item.name
    )

print("\n" + "=" * 65)
print("ALL DIRECTORIES")
print("=" * 65)

for folder in sorted(DATASET_DIR.rglob("*")):
    if folder.is_dir():
        print(folder.relative_to(DATASET_DIR))

print("\n" + "=" * 65)
print("ALL YAML FILES")
print("=" * 65)

for yaml_file in DATASET_DIR.rglob("*.yaml"):
    print("\nFILE:", yaml_file)

    with open(yaml_file, "r") as f:
        print(f.read())

print("\n" + "=" * 65)
print("SAMPLE FILES")
print("=" * 65)

for file in list(DATASET_DIR.rglob("*"))[:40]:
    if file.is_file():
        print(file.relative_to(DATASET_DIR))


DATASET DIR: /content/drive/MyDrive/Hazard_Detection/raw_datasets/doors/source_datasets/door_v3
EXISTS: True

TOP-LEVEL CONTENTS
[FILE] README.dataset.txt
[FILE] README.roboflow.txt
[FILE] data.yaml
[DIR]  test
[DIR]  train
[DIR]  valid

ALL DIRECTORIES
test
test/images
test/labels
train
train/images
train/labels
valid
valid/images
valid/labels

ALL YAML FILES

FILE: /content/drive/MyDrive/Hazard_Detection/raw_datasets/doors/source_datasets/door_v3/data.yaml
names:
- door
nc: 1
roboflow:
  license: CC BY 4.0
  project: door-jqeug
  url: https://universe.roboflow.com/object-detect-zjtnj/door-jqeug/dataset/3
  version: 3
  workspace: object-detect-zjtnj
test: ../test/images
train: ../train/images
val: ../valid/images


SAMPLE FILES
README.dataset.txt
README.roboflow.txt
data.yaml
valid/images/istockphoto-1003379164-612x612_jpg.rf.bf0d72d5a0b397281225b9c8e2198c0d.jpg
valid/images/istockphoto-1027914342-612x612_jpg.rf.36f2dde78d815fb4030d5361108b94c4.jpg
valid/images/istockphoto-1041825124

In [7]:

# ==========================================
# DOOR DATASET DIAGNOSTIC (CORRECTED)
# ==========================================

from collections import Counter

total_instances = 0

for split in ["train", "valid", "test"]:

    LABEL_DIR = DATASET_DIR / split / "labels"
    IMAGE_DIR = DATASET_DIR / split / "images"

    if not LABEL_DIR.exists():
        raise FileNotFoundError(LABEL_DIR)

    counts = Counter()
    images_with_labels = 0
    empty_labels = 0

    for label_file in LABEL_DIR.glob("*.txt"):

        lines = [
            line.strip()
            for line in label_file.read_text().splitlines()
            if line.strip()
        ]

        if lines:
            images_with_labels += 1
        else:
            empty_labels += 1

        for line in lines:
            class_id = int(line.split()[0])
            counts[class_id] += 1

    instances = sum(counts.values())
    total_instances += instances

    image_count = len(list(IMAGE_DIR.iterdir()))

    print("\n" + "=" * 65)
    print(split.upper())
    print("=" * 65)

    print("Images:", image_count)
    print("Images with annotations:", images_with_labels)
    print("Empty label files:", empty_labels)
    print("Door instances:", counts[0])

print("\n" + "=" * 65)
print("TOTAL DOOR INSTANCES:", total_instances)
print("TARGET:", 200)
print("=" * 65)



TRAIN
Images: 700
Images with annotations: 700
Empty label files: 0
Door instances: 715

VALID
Images: 200
Images with annotations: 200
Empty label files: 0
Door instances: 202

TEST
Images: 100
Images with annotations: 100
Empty label files: 0
Door instances: 103

TOTAL DOOR INSTANCES: 1020
TARGET: 200


In [8]:

# ==========================================
# CHECK EXACT WHOLE-IMAGE SELECTION
# ==========================================

import random
from collections import Counter

TARGETS = {
    "train": 140,
    "valid": 30,
    "test": 30
}

RANDOM_SEED = 42
random.seed(RANDOM_SEED)


def inspect_split(split):

    label_dir = DATASET_DIR / split / "labels"

    image_counts = []

    for label_file in label_dir.glob("*.txt"):

        lines = [
            line.strip()
            for line in label_file.read_text().splitlines()
            if line.strip()
        ]

        if lines:
            image_counts.append(
                (label_file.stem, len(lines))
            )

    return image_counts


def select_exact_images(image_counts, target):

    shuffled = image_counts.copy()
    random.shuffle(shuffled)

    # Dynamic programming: instance total -> selected images
    dp = {0: []}

    for stem, count in shuffled:

        for current_total, selected in list(dp.items()):

            new_total = current_total + count

            if new_total <= target and new_total not in dp:
                dp[new_total] = selected + [(stem, count)]

        if target in dp:
            return dp[target]

    return None


print("=" * 65)
print("WHOLE-IMAGE SELECTION FEASIBILITY")
print("=" * 65)

for split, target in TARGETS.items():

    image_counts = inspect_split(split)

    distribution = Counter(
        count for _, count in image_counts
    )

    selection = select_exact_images(
        image_counts,
        target
    )

    print(f"\n{split.upper()}")
    print("-" * 50)

    print("Available annotated images:", len(image_counts))
    print("Instance distribution:", dict(sorted(distribution.items())))
    print("Target instances:", target)

    if selection is None:
        print("EXACT WHOLE-IMAGE SELECTION: NOT POSSIBLE")
    else:
        print("EXACT WHOLE-IMAGE SELECTION: POSSIBLE")
        print("Selected images:", len(selection))
        print("Selected instances:", sum(n for _, n in selection))


WHOLE-IMAGE SELECTION FEASIBILITY

TRAIN
--------------------------------------------------
Available annotated images: 700
Instance distribution: {1: 686, 2: 13, 3: 1}
Target instances: 140
EXACT WHOLE-IMAGE SELECTION: POSSIBLE
Selected images: 137
Selected instances: 140

VALID
--------------------------------------------------
Available annotated images: 200
Instance distribution: {1: 198, 2: 2}
Target instances: 30
EXACT WHOLE-IMAGE SELECTION: POSSIBLE
Selected images: 28
Selected instances: 30

TEST
--------------------------------------------------
Available annotated images: 100
Instance distribution: {1: 98, 2: 1, 3: 1}
Target instances: 30
EXACT WHOLE-IMAGE SELECTION: POSSIBLE
Selected images: 30
Selected instances: 30


In [9]:

# ==========================================
# CREATE FINAL DOOR DATASET
# EXACTLY 200 INSTANCES
# ==========================================

import random
import shutil
import yaml

from pathlib import Path
from collections import Counter

# ------------------------------------------
# CONFIGURATION
# ------------------------------------------

FINAL_DIR = BASE_DIR / "final_dataset"

TARGETS = {
    "train": 140,
    "valid": 30,
    "test": 30
}

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

if FINAL_DIR.exists():
    raise FileExistsError(
        f"Final dataset already exists: {FINAL_DIR}\n"
        "Inspect it before proceeding. Do not overwrite."
    )


# ------------------------------------------
# SELECT EXACT WHOLE IMAGES
# ------------------------------------------

def select_exact_images(split, target):

    label_dir = DATASET_DIR / split / "labels"

    image_counts = []

    for label_file in label_dir.glob("*.txt"):

        lines = [
            line.strip()
            for line in label_file.read_text().splitlines()
            if line.strip()
        ]

        if lines:
            image_counts.append(
                (label_file.stem, len(lines))
            )

    shuffled = image_counts.copy()
    random.shuffle(shuffled)

    dp = {0: []}

    for stem, count in shuffled:

        for current_total, selected in list(dp.items()):

            new_total = current_total + count

            if new_total <= target and new_total not in dp:
                dp[new_total] = selected + [(stem, count)]

        if target in dp:
            return dp[target]

    raise RuntimeError(
        f"Could not achieve {target} instances in {split}"
    )


# ------------------------------------------
# CREATE OUTPUT DIRECTORIES
# ------------------------------------------

for split in TARGETS:

    (FINAL_DIR / split / "images").mkdir(
        parents=True,
        exist_ok=True
    )

    (FINAL_DIR / split / "labels").mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------
# COPY SELECTED IMAGES AND LABELS
# ------------------------------------------

selection_summary = {}

for split, target in TARGETS.items():

    print("\n" + "=" * 65)
    print("PROCESSING:", split.upper())
    print("=" * 65)

    selected = select_exact_images(split, target)

    selected_instances = 0

    for stem, expected_count in selected:

        source_label = (
            DATASET_DIR / split / "labels" / f"{stem}.txt"
        )

        # Find the corresponding image
        matching_images = list(
            (DATASET_DIR / split / "images").glob(stem + ".*")
        )

        if len(matching_images) != 1:
            raise RuntimeError(
                f"Expected exactly one image for {stem}, "
                f"found {len(matching_images)}"
            )

        source_image = matching_images[0]

        # Validate annotations
        lines = [
            line.strip()
            for line in source_label.read_text().splitlines()
            if line.strip()
        ]

        if len(lines) != expected_count:
            raise RuntimeError(
                f"Annotation count changed for {stem}"
            )

        # Copy image
        shutil.copy2(
            source_image,
            FINAL_DIR / split / "images" / source_image.name
        )

        # Copy complete annotation
        shutil.copy2(
            source_label,
            FINAL_DIR / split / "labels" / source_label.name
        )

        selected_instances += len(lines)

    if selected_instances != target:
        raise RuntimeError(
            f"{split}: Expected {target}, got {selected_instances}"
        )

    selection_summary[split] = {
        "images": len(selected),
        "instances": selected_instances
    }

    print("Selected images:", len(selected))
    print("Selected instances:", selected_instances)


# ------------------------------------------
# GENERATE DATA.YAML
# ------------------------------------------

final_config = {
    "path": str(FINAL_DIR),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 1,
    "names": ["door"]
}

with open(FINAL_DIR / "data.yaml", "w") as f:
    yaml.safe_dump(
        final_config,
        f,
        sort_keys=False
    )


print("\n" + "=" * 65)
print("DATASET CREATION COMPLETE")
print("=" * 65)

for split, info in selection_summary.items():
    print(
        f"{split.upper():<10} "
        f"Images: {info['images']:<5} "
        f"Instances: {info['instances']}"
    )

print(
    "TOTAL INSTANCES:",
    sum(x["instances"] for x in selection_summary.values())
)

print("OUTPUT:", FINAL_DIR)



PROCESSING: TRAIN
Selected images: 137
Selected instances: 140

PROCESSING: VALID
Selected images: 28
Selected instances: 30

PROCESSING: TEST
Selected images: 30
Selected instances: 30

DATASET CREATION COMPLETE
TRAIN      Images: 137   Instances: 140
VALID      Images: 28    Instances: 30
TEST       Images: 30    Instances: 30
TOTAL INSTANCES: 200
OUTPUT: /content/drive/MyDrive/Hazard_Detection/raw_datasets/doors/final_dataset


In [10]:

# ==========================================
# REMOVE OLD FINAL DATASET
# ==========================================

import shutil

FINAL_DIR = BASE_DIR / "final_dataset"

if FINAL_DIR.exists():

    print("Removing previous final dataset:")
    print(FINAL_DIR)

    shutil.rmtree(FINAL_DIR)

    print("Old final dataset deleted.")

else:
    print("No previous final dataset found.")

print("\nOriginal Roboflow download remains untouched.")


Removing previous final dataset:
/content/drive/MyDrive/Hazard_Detection/raw_datasets/doors/final_dataset
Old final dataset deleted.

Original Roboflow download remains untouched.


In [11]:

# ==========================================
# CREATE FINAL DOOR DATASET
# 400 INSTANCES
# ==========================================

import random
import shutil
import yaml

from collections import Counter

FINAL_DIR = BASE_DIR / "final_dataset"

TARGETS = {
    "train": 270,
    "valid": 30,
    "test": 100
}

random.seed(42)

if FINAL_DIR.exists():
    raise FileExistsError(
        "Final dataset already exists. "
        "Inspect it before proceeding."
    )


# ==========================================
# EXACT WHOLE-IMAGE SELECTION
# ==========================================

def select_exact_images(split, target):

    label_dir = DATASET_DIR / split / "labels"

    image_counts = []

    for label_file in label_dir.glob("*.txt"):

        lines = [
            line.strip()
            for line in label_file.read_text().splitlines()
            if line.strip()
        ]

        if lines:
            image_counts.append(
                (label_file.stem, len(lines))
            )

    random.shuffle(image_counts)

    dp = {0: []}

    for stem, count in image_counts:

        for current_total, selected in list(dp.items()):

            new_total = current_total + count

            if new_total <= target and new_total not in dp:

                dp[new_total] = (
                    selected + [(stem, count)]
                )

        if target in dp:
            return dp[target]

    raise RuntimeError(
        f"Cannot achieve {target} instances in {split}"
    )


# ==========================================
# CREATE DIRECTORIES
# ==========================================

for split in TARGETS:

    (FINAL_DIR / split / "images").mkdir(
        parents=True,
        exist_ok=True
    )

    (FINAL_DIR / split / "labels").mkdir(
        parents=True,
        exist_ok=True
    )


# ==========================================
# COPY SELECTED DATA
# ==========================================

summary = {}

for split, target in TARGETS.items():

    print("\n" + "=" * 65)
    print("PROCESSING:", split.upper())
    print("=" * 65)

    selected = select_exact_images(split, target)

    instance_count = 0

    for stem, expected_count in selected:

        source_label = (
            DATASET_DIR / split / "labels" / f"{stem}.txt"
        )

        matching_images = list(
            (DATASET_DIR / split / "images").glob(stem + ".*")
        )

        if len(matching_images) != 1:
            raise RuntimeError(
                f"Image matching error: {stem}"
            )

        source_image = matching_images[0]

        lines = [
            line.strip()
            for line in source_label.read_text().splitlines()
            if line.strip()
        ]

        if len(lines) != expected_count:
            raise RuntimeError(
                f"Annotation mismatch: {stem}"
            )

        # Copy complete image
        shutil.copy2(
            source_image,
            FINAL_DIR / split / "images" / source_image.name
        )

        # Copy complete annotations
        shutil.copy2(
            source_label,
            FINAL_DIR / split / "labels" / source_label.name
        )

        instance_count += len(lines)

    if instance_count != target:
        raise RuntimeError(
            f"{split}: Expected {target}, got {instance_count}"
        )

    summary[split] = {
        "images": len(selected),
        "instances": instance_count
    }

    print("Selected images:", len(selected))
    print("Selected instances:", instance_count)


# ==========================================
# GENERATE DATA.YAML
# ==========================================

config = {
    "path": str(FINAL_DIR),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 1,
    "names": ["door"]
}

with open(FINAL_DIR / "data.yaml", "w") as f:

    yaml.safe_dump(
        config,
        f,
        sort_keys=False
    )


# ==========================================
# SUMMARY
# ==========================================

print("\n" + "=" * 65)
print("DATASET CREATION COMPLETE")
print("=" * 65)

for split, info in summary.items():

    print(
        f"{split.upper():<10} "
        f"Images: {info['images']:<5} "
        f"Instances: {info['instances']}"
    )

print(
    "\nTOTAL:",
    sum(x["instances"] for x in summary.values())
)

print("OUTPUT:", FINAL_DIR)



PROCESSING: TRAIN
Selected images: 264
Selected instances: 270

PROCESSING: VALID
Selected images: 28
Selected instances: 30

PROCESSING: TEST
Selected images: 97
Selected instances: 100

DATASET CREATION COMPLETE
TRAIN      Images: 264   Instances: 270
VALID      Images: 28    Instances: 30
TEST       Images: 97    Instances: 100

TOTAL: 400
OUTPUT: /content/drive/MyDrive/Hazard_Detection/raw_datasets/doors/final_dataset


In [12]:

# ==========================================
# FINAL DATASET VERIFICATION
# ==========================================

total = 0
split_stems = {}

for split, target in TARGETS.items():

    image_dir = FINAL_DIR / split / "images"
    label_dir = FINAL_DIR / split / "labels"

    images = list(image_dir.iterdir())
    labels = list(label_dir.glob("*.txt"))

    image_stems = {p.stem for p in images}
    label_stems = {p.stem for p in labels}

    assert image_stems == label_stems, (
        f"Image-label mismatch: {split}"
    )

    instances = 0

    for label in labels:

        lines = [
            x.strip()
            for x in label.read_text().splitlines()
            if x.strip()
        ]

        for line in lines:

            parts = line.split()

            assert len(parts) == 5
            assert int(parts[0]) == 0

            instances += 1

    assert instances == target, (
        f"{split}: Expected {target}, got {instances}"
    )

    split_stems[split] = image_stems
    total += instances

    print(f"\n{split.upper()}")
    print("Images:", len(images))
    print("Instances:", instances)
    print("Target met:", instances == target)


# Cross-split overlap
assert split_stems["train"].isdisjoint(split_stems["valid"])
assert split_stems["train"].isdisjoint(split_stems["test"])
assert split_stems["valid"].isdisjoint(split_stems["test"])

assert total == 400

print("\n" + "=" * 65)
print("ALL CHECKS PASSED")
print("=" * 65)

print("Total instances:", total)
print("Cross-split overlap: NONE")
print("Class: 0 = door")
print("Location:", FINAL_DIR)


AssertionError: 

In [13]:

# ==========================================
# DIAGNOSE INVALID DOOR ANNOTATIONS
# ==========================================

for split in ["train", "valid", "test"]:

    label_dir = FINAL_DIR / split / "labels"

    print("\n" + "=" * 65)
    print("SPLIT:", split.upper())
    print("=" * 65)

    invalid_count = 0

    for label_file in label_dir.glob("*.txt"):

        lines = label_file.read_text().splitlines()

        for line_number, line in enumerate(lines, start=1):

            line = line.strip()

            if not line:
                continue

            parts = line.split()

            if len(parts) != 5:

                invalid_count += 1

                print("\nFILE:", label_file.name)
                print("LINE:", line_number)
                print("VALUES:", repr(line))
                print("NUMBER OF VALUES:", len(parts))

                if invalid_count >= 10:
                    break

        if invalid_count >= 10:
            break

    print("\nInvalid lines found (up to 10 displayed):", invalid_count)



SPLIT: TRAIN

FILE: istockphoto-177330083-612x612_jpg.rf.fc52a9bcdb660da38af2726975e3e96a.txt
LINE: 1
VALUES: '0 0.58097385625 0.9150326796874999 0.58097385625 0.0939542484375 0.5646339874999999 0.0939542484375 0.24328322500000002 0.15931372500000002 0.2360196078125 0.8428643796875 0.5610021781250001 0.919117646875 0.58097385625 0.9150326796874999'
NUMBER OF VALUES: 15

FILE: istockphoto-1465499623-612x612_jpg.rf.6cd30d71718704f7f06940050e0c2ae3.txt
LINE: 1
VALUES: '0 0.6964877453125 0.3458611109375 0.353350490625 0.34449836562499997 0.345178921875 0.7883986921874999 0.6964877453125 0.7856748359375 0.6964877453125 0.3458611109375'
NUMBER OF VALUES: 11

FILE: istockphoto-480491394-612x612_jpg.rf.147378663a5530ccaa54ceb4832d832a.txt
LINE: 1
VALUES: '0 0.32952040312499997 0.105624934375 0.3431372546875 0.10399334375 0.3295200546875 0.10572703750000001 0.3267973859375 0.9026622296875001 0.6386160125 0.9761514140625 0.6304460781250001 0.045757071875 0.32518015 0.10601130781250001 0.3295212

In [14]:

# ==========================================
# CONVERT YOLO POLYGONS TO YOLO BOUNDING BOXES
# ==========================================

import math
import os

from pathlib import Path

converted = 0
already_boxes = 0
total_instances = 0

for split, target in TARGETS.items():

    label_dir = FINAL_DIR / split / "labels"

    print("\n" + "=" * 65)
    print("CONVERTING:", split.upper())
    print("=" * 65)

    split_instances = 0

    for label_file in label_dir.glob("*.txt"):

        original_lines = label_file.read_text().splitlines()
        converted_lines = []

        for line_number, line in enumerate(original_lines, 1):

            line = line.strip()

            if not line:
                continue

            parts = line.split()

            class_id = int(parts[0])

            if class_id != 0:
                raise ValueError(
                    f"Unexpected class in {label_file.name}"
                )

            values = [float(x) for x in parts[1:]]

            # Already a YOLO bounding box
            if len(parts) == 5:

                already_boxes += 1
                converted_lines.append(line)
                split_instances += 1
                continue

            # Polygon must contain at least 3 coordinate pairs
            if len(values) < 6 or len(values) % 2 != 0:

                raise ValueError(
                    f"Invalid polygon: {label_file.name}, "
                    f"line {line_number}"
                )

            if not all(math.isfinite(v) for v in values):
                raise ValueError(
                    f"Non-finite coordinate: {label_file.name}"
                )

            xs = values[0::2]
            ys = values[1::2]

            if any(v < 0 or v > 1 for v in xs + ys):
                raise ValueError(
                    f"Coordinates outside [0,1]: "
                    f"{label_file.name}"
                )

            # Polygon boundaries
            x_min = min(xs)
            x_max = max(xs)

            y_min = min(ys)
            y_max = max(ys)

            # YOLO bounding box
            x_center = (x_min + x_max) / 2
            y_center = (y_min + y_max) / 2

            width = x_max - x_min
            height = y_max - y_min

            bbox_line = (
                f"0 {x_center:.10f} {y_center:.10f} "
                f"{width:.10f} {height:.10f}"
            )

            converted_lines.append(bbox_line)

            converted += 1
            split_instances += 1

        # Replace file atomically; no duplicate dataset created
        temp_file = label_file.with_name(
            label_file.name + ".tmp"
        )

        temp_file.write_text(
            "\n".join(converted_lines) + "\n"
        )

        os.replace(temp_file, label_file)

    if split_instances != target:
        raise RuntimeError(
            f"{split}: Expected {target}, got {split_instances}"
        )

    total_instances += split_instances

    print("Converted polygons:", converted)
    print("Instances in split:", split_instances)
    print("Target:", target)

print("\n" + "=" * 65)
print("CONVERSION COMPLETE")
print("=" * 65)

print("Polygons converted:", converted)
print("Already bounding boxes:", already_boxes)
print("Total instances:", total_instances)

assert total_instances == 400

print("All 400 instances preserved.")



CONVERTING: TRAIN
Converted polygons: 9
Instances in split: 270
Target: 270

CONVERTING: VALID
Converted polygons: 9
Instances in split: 30
Target: 30

CONVERTING: TEST
Converted polygons: 10
Instances in split: 100
Target: 100

CONVERSION COMPLETE
Polygons converted: 10
Already bounding boxes: 390
Total instances: 400
All 400 instances preserved.


In [15]:

# ==========================================
# VERIFY FINAL YOLO DETECTION FORMAT
# ==========================================

total = 0
split_stems = {}

for split, target in TARGETS.items():

    image_dir = FINAL_DIR / split / "images"
    label_dir = FINAL_DIR / split / "labels"

    images = list(image_dir.iterdir())
    labels = list(label_dir.glob("*.txt"))

    assert {p.stem for p in images} == {
        p.stem for p in labels
    }, f"Image-label mismatch: {split}"

    instances = 0

    for label in labels:

        for line in label.read_text().splitlines():

            if not line.strip():
                continue

            parts = line.split()

            assert len(parts) == 5, (
                f"Invalid YOLO detection format: {label.name}"
            )

            assert int(parts[0]) == 0

            coords = [float(x) for x in parts[1:]]

            assert all(
                math.isfinite(x) and 0 <= x <= 1
                for x in coords
            )

            x, y, w, h = coords

            assert w > 0 and h > 0
            assert x - w/2 >= -1e-8
            assert y - h/2 >= -1e-8
            assert x + w/2 <= 1 + 1e-8
            assert y + h/2 <= 1 + 1e-8

            instances += 1

    assert instances == target

    split_stems[split] = {p.stem for p in images}
    total += instances

    print(f"\n{split.upper()}")
    print("Images:", len(images))
    print("Instances:", instances)
    print("YOLO format: VALID")

assert split_stems["train"].isdisjoint(split_stems["valid"])
assert split_stems["train"].isdisjoint(split_stems["test"])
assert split_stems["valid"].isdisjoint(split_stems["test"])

assert total == 400

print("\n" + "=" * 65)
print("ALL CHECKS PASSED")
print("=" * 65)

print("Total instances:", total)
print("Train: 270")
print("Validation: 30")
print("Test: 100")
print("Class: 0 = door")
print("Format: YOLO Detection")
print("Cross-split overlap: NONE")



TRAIN
Images: 264
Instances: 270
YOLO format: VALID

VALID
Images: 28
Instances: 30
YOLO format: VALID

TEST
Images: 97
Instances: 100
YOLO format: VALID

ALL CHECKS PASSED
Total instances: 400
Train: 270
Validation: 30
Test: 100
Class: 0 = door
Format: YOLO Detection
Cross-split overlap: NONE


In [16]:
# DELETE ORIGINAL DOOR DOWNLOAD

import shutil
SOURCE_TO_DELETE = BASE_DIR / "source_datasets" / "door_v3"
assert (FINAL_DIR / "data.yaml").exists()

assert sum(
    len(list((FINAL_DIR / split / "labels").glob("*.txt")))
    for split in ["train", "valid", "test"]
) > 0

confirmation = input("Type DELETE SOURCE to continue: ")

if confirmation == "DELETE SOURCE":
    shutil.rmtree(SOURCE_TO_DELETE)
    print("Original download deleted.")
    print("Final dataset preserved:", FINAL_DIR)
else:
    print("Cancelled.")

Type DELETE SOURCE to continue: DELETE SOURCE
Original download deleted.
Final dataset preserved: /content/drive/MyDrive/Hazard_Detection/raw_datasets/doors/final_dataset


In [18]:

from pathlib import Path
import yaml
import os

# ==========================================
# REMAP DOOR: CLASS 0 -> CLASS 17
# ==========================================

FINAL_DIR = Path(
    "/content/drive/MyDrive/Hazard_Detection/"
    "raw_datasets/doors/final_dataset"
)

TARGETS = {
    "train": 270,
    "valid": 30,
    "test": 100
}

MASTER_NAMES = {
    0: "person",
    1: "two_wheeler",
    2: "car",
    3: "auto_rickshaw",
    4: "large_vehicle",
    5: "large_animal",
    6: "dog",
    7: "pothole",
    8: "open_drain",
    9: "speed_breaker",
    10: "stairs_up",
    11: "stairs_down",
    12: "curb_or_step",
    13: "obstacle",
    14: "vendor_stall",
    15: "bench",
    16: "chair",
    17: "door",
    18: "sports_balls",
    19: "traffic_lights",
    20: "other_hazards"
}

# ==========================================
# PREFLIGHT CHECKS
# ==========================================

assert FINAL_DIR.exists(), f"Dataset not found: {FINAL_DIR}"

print("PREFLIGHT")
print("=" * 60)

for split, expected in TARGETS.items():
    label_dir = FINAL_DIR / split / "labels"
    assert label_dir.exists(), f"Missing labels folder: {label_dir}"

    label_files = list(label_dir.glob("*.txt"))
    instance_count = 0

    for file in label_files:
        for line in file.read_text().splitlines():
            if not line.strip():
                continue

            parts = line.split()

            assert len(parts) == 5, (
                f"Unexpected annotation format in {file.name}: {line}"
            )

            class_id = int(float(parts[0]))

            assert class_id in (0, 17), (
                f"Unexpected class ID {class_id} in {file.name}"
            )

            instance_count += 1

    assert instance_count == expected, (
        f"{split}: expected {expected}, found {instance_count}"
    )

    print(f"{split}: {instance_count} instances verified")

# ==========================================
# REMAP ANNOTATIONS
# ==========================================

for split in TARGETS:
    label_dir = FINAL_DIR / split / "labels"

    for file in label_dir.glob("*.txt"):
        lines = file.read_text().splitlines()
        updated = []

        for line in lines:
            if not line.strip():
                continue

            parts = line.split()
            parts[0] = "17"
            updated.append(" ".join(parts))

        temp_file = file.with_suffix(".tmp")
        temp_file.write_text(
            "\n".join(updated) + ("\n" if updated else "")
        )
        os.replace(temp_file, file)

# ==========================================
# UPDATE DATA.YAML
# ==========================================

data_yaml = {
    "path": str(FINAL_DIR),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 21,
    "names": MASTER_NAMES
}

with open(FINAL_DIR / "data.yaml", "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print("\nRemapping and YAML update complete!")

# ==========================================
# FINAL VERIFICATION
# ==========================================

print("\nFINAL VERIFICATION")
print("=" * 60)

for split, expected in TARGETS.items():
    label_dir = FINAL_DIR / split / "labels"
    count = 0

    for file in label_dir.glob("*.txt"):
        for line in file.read_text().splitlines():
            if not line.strip():
                continue

            parts = line.split()
            assert len(parts) == 5
            assert int(parts[0]) == 17

            coords = [float(x) for x in parts[1:]]
            assert all(0 <= x <= 1 for x in coords)
            assert coords[2] > 0 and coords[3] > 0

            count += 1

    assert count == expected
    print(f"{split}: {count} door instances, all class ID 17")

print("\nMASTER MAPPING:")
print(f"nc: 21")
print(f"Class 17: {MASTER_NAMES[17]}")
print("\nALL CHECKS PASSED.")


PREFLIGHT
train: 270 instances verified
valid: 30 instances verified
test: 100 instances verified

Remapping and YAML update complete!

FINAL VERIFICATION
train: 270 door instances, all class ID 17
valid: 30 door instances, all class ID 17
test: 100 door instances, all class ID 17

MASTER MAPPING:
nc: 21
Class 17: door

ALL CHECKS PASSED.


In [4]:

from pathlib import Path
from collections import Counter
import yaml
import os

# ============================================================
# DOORS DATASET - READ-ONLY AUDIT
# ============================================================

PROJECT_ROOT = Path("/content/drive/MyDrive/Hazard_Detection")

MASTER_YAML = PROJECT_ROOT / "master_class" / "classes.yaml"
DOORS_DATASET = PROJECT_ROOT / "raw_datasets" / "doors" / "final_dataset"

TARGET_CLASS = 17

print("=" * 75)
print("DOORS DATASET AUDIT")
print("=" * 75)

# Load master classes
with open(MASTER_YAML, "r") as f:
    master = yaml.safe_load(f)

names = master["names"]

if isinstance(names, dict):
    door_name = names[TARGET_CLASS]
else:
    door_name = names[TARGET_CLASS]

print(f"\nTarget class: {TARGET_CLASS} ({door_name})")
print(f"Dataset: {DOORS_DATASET}")

if not DOORS_DATASET.exists():
    raise FileNotFoundError(f"Dataset not found: {DOORS_DATASET}")

# Find supported split folders
split_aliases = {
    "train": ["train"],
    "valid": ["valid", "val"],
    "test": ["test"]
}

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

total_images = 0
total_door_instances = 0
total_annotations = 0

for split, aliases in split_aliases.items():

    image_dirs = []
    label_dirs = []

    for alias in aliases:
        candidates = [
            DOORS_DATASET / "images" / alias,
            DOORS_DATASET / alias / "images"
        ]

        for candidate in candidates:
            if candidate.exists():
                image_dirs.append(candidate)

        candidates = [
            DOORS_DATASET / "labels" / alias,
            DOORS_DATASET / alias / "labels"
        ]

        for candidate in candidates:
            if candidate.exists():
                label_dirs.append(candidate)

    if not image_dirs and not label_dirs:
        continue

    images = {}
    for folder in image_dirs:
        for p in folder.rglob("*"):
            if p.is_file() and p.suffix.lower() in image_extensions:
                images[p.stem] = p

    labels = {}
    for folder in label_dirs:
        for p in folder.rglob("*.txt"):
            labels[p.stem] = p

    door_instances = 0
    annotations = 0
    malformed = 0
    invalid_boxes = 0
    other_classes = Counter()
    labelled_images = 0

    for stem, label_path in labels.items():

        try:
            lines = label_path.read_text().splitlines()
        except Exception:
            malformed += 1
            continue

        has_annotation = False

        for line in lines:
            parts = line.strip().split()

            if len(parts) != 5:
                if line.strip():
                    malformed += 1
                continue

            try:
                cls = int(float(parts[0]))
                x, y, w, h = map(float, parts[1:])
            except Exception:
                malformed += 1
                continue

            if not (0 <= x <= 1 and 0 <= y <= 1 and
                    0 < w <= 1 and 0 < h <= 1):
                invalid_boxes += 1

            annotations += 1
            other_classes[cls] += 1

            if cls == TARGET_CLASS:
                door_instances += 1
                has_annotation = True

        if has_annotation:
            labelled_images += 1

    split_images = len(images)

    total_images += split_images
    total_door_instances += door_instances
    total_annotations += annotations

    print(f"\n{'-' * 65}")
    print(f"SPLIT: {split.upper()}")
    print(f"{'-' * 65}")
    print(f"Images:                 {split_images}")
    print(f"Label files:            {len(labels)}")
    print(f"Images with doors:      {labelled_images}")
    print(f"Door instances:         {door_instances}")
    print(f"All annotations:        {annotations}")
    print(f"Malformed lines/files:  {malformed}")
    print(f"Invalid bounding boxes: {invalid_boxes}")
    print(f"Class IDs found:        {dict(other_classes)}")

print("\n" + "=" * 75)
print("OVERALL TOTALS")
print("=" * 75)
print(f"Images:          {total_images}")
print(f"Door instances:  {total_door_instances}")
print(f"Annotations:     {total_annotations}")
print("=" * 75)

print("\nAUDIT COMPLETE - NO FILES WERE MODIFIED.")


DOORS DATASET AUDIT

Target class: 17 (door)
Dataset: /content/drive/MyDrive/Hazard_Detection/raw_datasets/doors/final_dataset

-----------------------------------------------------------------
SPLIT: TRAIN
-----------------------------------------------------------------
Images:                 264
Label files:            264
Images with doors:      264
Door instances:         270
All annotations:        270
Malformed lines/files:  0
Invalid bounding boxes: 0
Class IDs found:        {17: 270}

-----------------------------------------------------------------
SPLIT: VALID
-----------------------------------------------------------------
Images:                 28
Label files:            28
Images with doors:      28
Door instances:         30
All annotations:        30
Malformed lines/files:  0
Invalid bounding boxes: 0
Class IDs found:        {17: 30}

-----------------------------------------------------------------
SPLIT: TEST
--------------------------------------------------------

In [5]:

# ============================================================
# DOWNLOAD KAGGLE DOORS DATASET - INSPECTION ONLY
# ============================================================

!pip -q install kagglehub

import kagglehub
import shutil
from pathlib import Path
from collections import Counter

PROJECT_ROOT = Path("/content/drive/MyDrive/Hazard_Detection")

STAGING = (
    PROJECT_ROOT
    / "raw_datasets"
    / "doors"
    / "kaggle_doors_detection_source"
)

HANDLE = "sayedmohamed1/doors-detection"

print("Downloading Kaggle dataset...")

download_path = Path(kagglehub.dataset_download(HANDLE))

print("Downloaded to:", download_path)

# Copy to persistent Google Drive staging folder
if download_path.resolve() != STAGING.resolve():
    if STAGING.exists():
        print("\nStaging folder already exists.")
        print("Keeping existing folder; not overwriting it.")
    else:
        shutil.copytree(download_path, STAGING)

print("\nPersistent source folder:", STAGING)

# ============================================================
# INSPECT FILE STRUCTURE
# ============================================================

print("\n" + "=" * 70)
print("KAGGLE DOORS DATASET - FILE AUDIT")
print("=" * 70)

image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

counts = Counter()
examples = {}

for p in STAGING.rglob("*"):
    if not p.is_file():
        continue

    ext = p.suffix.lower()

    if ext in image_exts:
        counts["images"] += 1
        examples.setdefault("images", []).append(str(p.relative_to(STAGING)))

    elif ext == ".txt":
        counts["txt_files"] += 1
        examples.setdefault("txt_files", []).append(str(p.relative_to(STAGING)))

    elif ext in {".yaml", ".yml"}:
        counts["yaml_files"] += 1
        examples.setdefault("yaml_files", []).append(str(p.relative_to(STAGING)))

    elif ext == ".json":
        counts["json_files"] += 1
        examples.setdefault("json_files", []).append(str(p.relative_to(STAGING)))

    elif ext == ".xml":
        counts["xml_files"] += 1
        examples.setdefault("xml_files", []).append(str(p.relative_to(STAGING)))

    elif ext == ".csv":
        counts["csv_files"] += 1
        examples.setdefault("csv_files", []).append(str(p.relative_to(STAGING)))

print("\nFILE COUNTS:")
for key, value in counts.items():
    print(f"{key:15}: {value}")

print("\n" + "=" * 70)
print("FOLDER STRUCTURE (first 100 entries)")
print("=" * 70)

for p in sorted(STAGING.rglob("*")):
    if p.is_file():
        print(p.relative_to(STAGING))
        if sum(1 for _ in []) > 100:
            break

print("\n" + "=" * 70)
print("SAMPLE FILES")
print("=" * 70)

for category, files in examples.items():
    print(f"\n{category.upper()}:")
    for name in files[:10]:
        print(" ", name)

print("\nINSPECTION DOWNLOAD COMPLETE.")
print("No existing dataset files were modified.")


100%|██████████| 62.9M/62.9M [00:00<00:00, 110MB/s] 

Extracting files...


Downloaded to: /root/.cache/kagglehub/datasets/sayedmohamed1/doors-detection/versions/1

Persistent source folder: /content/drive/MyDrive/Hazard_Detection/raw_datasets/doors/kaggle_doors_detection_source

KAGGLE DOORS DATASET - FILE AUDIT

FILE COUNTS:
txt_files      : 477
images         : 477

FOLDER STRUCTURE (first 100 entries)
train/images/IMG_20190624_104925.jpg
train/images/IMG_20190624_104950.jpg
train/images/IMG_20190624_104955.jpg
train/images/IMG_20190624_105015.jpg
train/images/IMG_20190624_105116.jpg
train/images/IMG_20190624_105124.jpg
train/images/IMG_20190624_105132.jpg
train/images/IMG_20190624_105503.jpg
train/images/IMG_20190624_105512.jpg
train/images/IMG_20190624_105521.jpg
train/images/IMG_20190624_105531.jpg
train/images/IMG_20190624_105536.jpg
train/images/IMG_20190624_105651.jpg
train/images/IMG_20190624_105702.jpg
train/images/IMG_20190624_105712.jpg
train/images/IMG_20190624_105725.jpg
train/images/IMG_20190624_105727.jpg
train/images/IMG_20190624_110107.jpg
t

In [6]:

# ============================================================
# KAGGLE DOORS - ANNOTATION + CROSS-SOURCE DUPLICATE AUDIT
# READ ONLY - DOES NOT MODIFY DATA
# ============================================================

from pathlib import Path
from collections import Counter
import hashlib

PROJECT_ROOT = Path("/content/drive/MyDrive/Hazard_Detection")

SOURCE = PROJECT_ROOT / "raw_datasets/doors/kaggle_doors_detection_source"
EXISTING = PROJECT_ROOT / "raw_datasets/doors/final_dataset"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def get_split_files(root, split):
    # Supports train/images, images/train, etc.
    image_dirs = [
        root / split / "images",
        root / "images" / split
    ]
    label_dirs = [
        root / split / "labels",
        root / "labels" / split
    ]

    images = {}
    labels = {}

    for folder in image_dirs:
        if folder.exists():
            for p in folder.rglob("*"):
                if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
                    images[p.stem] = p

    for folder in label_dirs:
        if folder.exists():
            for p in folder.rglob("*.txt"):
                labels[p.stem] = p

    return images, labels

print("=" * 75)
print("KAGGLE DOORS - DETAILED AUDIT")
print("=" * 75)

source_records = []
source_hashes = {}
existing_hashes = {}

# ------------------------------------------------------------
# 1. Hash existing final dataset images
# ------------------------------------------------------------

print("\nHashing existing final_dataset images...")

existing_splits = ["train", "valid", "val", "test"]

for split in existing_splits:
    images, _ = get_split_files(EXISTING, split)

    for stem, path in images.items():
        digest = sha256(path)
        existing_hashes.setdefault(digest, []).append(
            (split, str(path))
        )

print("Existing unique image hashes:", len(existing_hashes))

# ------------------------------------------------------------
# 2. Audit Kaggle train and validation
# ------------------------------------------------------------

overall_classes = Counter()
source_duplicates = []
existing_duplicates = []
split_stats = {}

for split in ["train", "val"]:

    images, labels = get_split_files(SOURCE, split)

    class_counts = Counter()
    malformed = 0
    invalid_boxes = 0
    empty_labels = 0
    missing_labels = []
    unmatched_labels = []
    valid_images = 0
    total_annotations = 0

    print("\n" + "-" * 65)
    print("SOURCE SPLIT:", split.upper())
    print("-" * 65)

    for stem, image_path in images.items():

        label_path = labels.get(stem)

        if label_path is None:
            missing_labels.append(stem)
            continue

        valid_images += 1

        digest = sha256(image_path)

        # Compare against existing final dataset
        if digest in existing_hashes:
            existing_duplicates.append({
                "source_split": split,
                "source_image": str(image_path),
                "existing_matches": existing_hashes[digest]
            })

        # Compare across Kaggle source splits
        if digest in source_hashes:
            source_duplicates.append({
                "source_split": split,
                "source_image": str(image_path),
                "previous_match": source_hashes[digest]
            })
        else:
            source_hashes[digest] = (split, str(image_path))

        lines = label_path.read_text(
            encoding="utf-8", errors="replace"
        ).splitlines()

        if not any(line.strip() for line in lines):
            empty_labels += 1

        for line in lines:
            parts = line.strip().split()

            if not parts:
                continue

            if len(parts) != 5:
                malformed += 1
                continue

            try:
                cls_float = float(parts[0])
                cls = int(cls_float)

                if cls_float != cls:
                    raise ValueError("Non-integer class ID")

                x, y, w, h = map(float, parts[1:])

                if not all(map(lambda v: abs(v) < float("inf"),
                               [x, y, w, h])):
                    raise ValueError("Non-finite box")

            except Exception:
                malformed += 1
                continue

            total_annotations += 1
            class_counts[cls] += 1
            overall_classes[cls] += 1

            if not (
                0 <= x <= 1 and
                0 <= y <= 1 and
                0 < w <= 1 and
                0 < h <= 1 and
                x - w / 2 >= -1e-6 and
                y - h / 2 >= -1e-6 and
                x + w / 2 <= 1 + 1e-6 and
                y + h / 2 <= 1 + 1e-6
            ):
                invalid_boxes += 1

    for stem in labels:
        if stem not in images:
            unmatched_labels.append(stem)

    split_stats[split] = {
        "images": len(images),
        "paired_images": valid_images,
        "labels": len(labels),
        "annotations": total_annotations,
        "classes": dict(class_counts),
        "malformed": malformed,
        "invalid_boxes": invalid_boxes,
        "empty_labels": empty_labels,
        "missing_labels": len(missing_labels),
        "unmatched_labels": len(unmatched_labels)
    }

    print("Images:", len(images))
    print("Paired image-label files:", valid_images)
    print("Label files:", len(labels))
    print("Valid annotations:", total_annotations)
    print("Class instance counts:", dict(class_counts))
    print("Malformed annotation lines:", malformed)
    print("Invalid bounding boxes:", invalid_boxes)
    print("Empty label files:", empty_labels)
    print("Images missing labels:", len(missing_labels))
    print("Labels missing images:", len(unmatched_labels))

# ------------------------------------------------------------
# 3. Final duplicate report
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("CROSS-SOURCE DUPLICATE RESULTS")
print("=" * 75)

print("Kaggle images matching existing final_dataset:",
      len(existing_duplicates))

print("Duplicates within Kaggle source across splits:",
      len(source_duplicates))

if existing_duplicates:
    print("\nEXAMPLES MATCHING EXISTING DATA:")
    for item in existing_duplicates[:10]:
        print(item["source_split"], "->", item["source_image"])
        print("Existing matches:", item["existing_matches"])

if source_duplicates:
    print("\nEXAMPLES OF KAGGLE INTERNAL DUPLICATES:")
    for item in source_duplicates[:10]:
        print(item)

# ------------------------------------------------------------
# 4. Overall summary
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("OVERALL KAGGLE CLASS COUNTS")
print("=" * 75)

for cls, count in sorted(overall_classes.items()):
    print(f"Class {cls}: {count} instances")

print("\n" + "=" * 75)
print("AUDIT COMPLETE - NO FILES MODIFIED")
print("=" * 75)


KAGGLE DOORS - DETAILED AUDIT

Hashing existing final_dataset images...
Existing unique image hashes: 389

-----------------------------------------------------------------
SOURCE SPLIT: TRAIN
-----------------------------------------------------------------
Images: 429
Paired image-label files: 429
Label files: 429
Valid annotations: 423
Class instance counts: {0: 423}
Malformed annotation lines: 0
Invalid bounding boxes: 0
Empty label files: 159
Images missing labels: 0
Labels missing images: 0

-----------------------------------------------------------------
SOURCE SPLIT: VAL
-----------------------------------------------------------------
Images: 48
Paired image-label files: 48
Label files: 48
Valid annotations: 51
Class instance counts: {0: 51}
Malformed annotation lines: 0
Invalid bounding boxes: 0
Empty label files: 13
Images missing labels: 0
Labels missing images: 0

CROSS-SOURCE DUPLICATE RESULTS
Kaggle images matching existing final_dataset: 0
Duplicates within Kaggle sour

In [7]:

# ================================================================
# BUILD UPDATED DOORS DATASET
# Target additions:
#   Train      + approximately 50 door instances
#   Validation + approximately 70 door instances
#   Test       + 0 door instances
#
# Original final_dataset is NEVER modified.
# Kaggle class 0 is remapped to master class 17.
# ================================================================

import os
import shutil
import hashlib
from pathlib import Path
from collections import Counter

# ---------------- CONFIGURATION ----------------

BASE = Path(
    "/content/drive/MyDrive/Hazard_Detection/raw_datasets/doors"
)

EXISTING = BASE / "final_dataset"

SOURCE = BASE / "kaggle_doors_detection_source"

OUTPUT = BASE / "updated_dataset"

DOOR_CLASS_ID = 17
SOURCE_CLASS_ID = 0

TRAIN_TARGET = 50
VALID_TARGET = 70

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}

# ------------------------------------------------
# SAFETY CHECKS
# ------------------------------------------------

if not EXISTING.exists():
    raise FileNotFoundError(f"Existing dataset not found: {EXISTING}")

if not SOURCE.exists():
    raise FileNotFoundError(f"Kaggle source not found: {SOURCE}")

if OUTPUT.exists():
    raise FileExistsError(
        f"\nOutput folder already exists:\n{OUTPUT}\n\n"
        "Rename or remove updated_dataset before running again. "
        "The script will not overwrite an existing output."
    )

print("=" * 65)
print("BUILDING UPDATED DOORS DATASET")
print("=" * 65)


# ------------------------------------------------
# HELPER FUNCTIONS
# ------------------------------------------------

def get_images(folder):
    return sorted([
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ])


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()


def read_door_labels(label_path, expected_class):
    """
    Returns valid annotation lines and their count.
    Rejects malformed lines, invalid boxes and unexpected classes.
    """

    valid_lines = []

    if not label_path.exists():
        raise FileNotFoundError(f"Missing label: {label_path}")

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line_number, line in enumerate(lines, 1):

        line = line.strip()

        if not line:
            continue

        parts = line.split()

        if len(parts) != 5:
            raise ValueError(
                f"Malformed annotation: {label_path}, line {line_number}"
            )

        try:
            class_id = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:])
        except ValueError:
            raise ValueError(
                f"Invalid annotation values: {label_path}, line {line_number}"
            )

        if class_id != expected_class:
            raise ValueError(
                f"Unexpected class {class_id} in {label_path}. "
                f"Expected class {expected_class}."
            )

        if not (
            0 <= x <= 1 and
            0 <= y <= 1 and
            0 < w <= 1 and
            0 < h <= 1 and
            x - w / 2 >= -1e-6 and
            x + w / 2 <= 1 + 1e-6 and
            y - h / 2 >= -1e-6 and
            y + h / 2 <= 1 + 1e-6
        ):
            raise ValueError(
                f"Invalid bounding box: {label_path}, line {line_number}"
            )

        # Remap source class to master class.
        valid_lines.append(
            f"{DOOR_CLASS_ID} {x} {y} {w} {h}\n"
        )

    return valid_lines


def collect_source_images(split_name):
    """
    Collects non-empty, correctly labelled Kaggle images.
    """

    image_dir = SOURCE / split_name / "images"
    label_dir = SOURCE / split_name / "labels"

    if not image_dir.exists() or not label_dir.exists():
        raise FileNotFoundError(
            f"Missing images or labels directory for {split_name}"
        )

    records = []

    for image_path in get_images(image_dir):

        label_path = label_dir / (image_path.stem + ".txt")

        if not label_path.exists():
            raise FileNotFoundError(
                f"Missing label for image: {image_path}"
            )

        labels = read_door_labels(label_path, SOURCE_CLASS_ID)

        # Ignore images without door annotations.
        if not labels:
            continue

        records.append({
            "image": image_path,
            "label": label_path,
            "labels": labels,
            "count": len(labels),
            "hash": sha256(image_path),
            "source_split": split_name
        })

    return records


def select_closest(records, target):
    """
    Selects whole images whose total annotation count is the
    smallest possible value >= target.

    No annotations are removed from selected images.
    """

    if target <= 0:
        return []

    # Dynamic programming: total instances -> selected record indices.
    dp = {0: []}

    for i, record in enumerate(records):

        weight = record["count"]
        updated = dict(dp)

        for total, indices in dp.items():

            new_total = total + weight

            if new_total not in updated:
                updated[new_total] = indices + [i]

        dp = updated

    possible = [total for total in dp if total >= target]

    if not possible:
        raise RuntimeError(
            f"Not enough source instances to reach target {target}. "
            f"Available: {sum(r['count'] for r in records)}"
        )

    best_total = min(possible)

    return [records[i] for i in dp[best_total]]


def copy_existing_split(split_name, output_split):
    """
    Copies the existing dataset without changing its labels.
    """

    image_dir = EXISTING / split_name / "images"
    label_dir = EXISTING / split_name / "labels"

    if not image_dir.exists() or not label_dir.exists():
        raise FileNotFoundError(
            f"Existing split missing: {split_name}"
        )

    count = 0

    for image_path in get_images(image_dir):

        label_path = label_dir / (image_path.stem + ".txt")

        if not label_path.exists():
            raise FileNotFoundError(
                f"Existing image has no label: {image_path}"
            )

        destination_name = "existing_" + image_path.name

        shutil.copy2(
            image_path,
            output_split / "images" / destination_name
        )

        shutil.copy2(
            label_path,
            output_split / "labels" /
            (Path(destination_name).stem + ".txt")
        )

        count += 1

    return count


def copy_kaggle_records(records, output_split):
    """
    Copies images and remapped labels.
    """

    for i, record in enumerate(records):

        image_path = record["image"]

        destination_name = (
            f"kaggle_{record['source_split']}_{i:05d}"
            + image_path.suffix.lower()
        )

        shutil.copy2(
            image_path,
            output_split / "images" / destination_name
        )

        label_destination = (
            output_split / "labels" /
            (Path(destination_name).stem + ".txt")
        )

        with open(label_destination, "w") as f:
            f.writelines(record["labels"])


def audit_output_split(split_name):
    """
    Counts images and door instances in the finished split.
    """

    image_dir = OUTPUT / split_name / "images"
    label_dir = OUTPUT / split_name / "labels"

    images = get_images(image_dir)

    total_instances = 0
    hashes = set()

    for image_path in images:

        label_path = label_dir / (image_path.stem + ".txt")

        if not label_path.exists():
            raise RuntimeError(f"Missing output label: {label_path}")

        with open(label_path, "r") as f:
            lines = [line.strip() for line in f if line.strip()]

        for line in lines:
            parts = line.split()

            if len(parts) != 5 or int(float(parts[0])) != DOOR_CLASS_ID:
                raise RuntimeError(
                    f"Invalid output annotation: {label_path}"
                )

        total_instances += len(lines)

        image_hash = sha256(image_path)

        if image_hash in hashes:
            raise RuntimeError(
                f"Duplicate image within output split: {image_path}"
            )

        hashes.add(image_hash)

    label_files = list(label_dir.glob("*.txt"))

    if len(images) != len(label_files):
        raise RuntimeError(
            f"Image-label count mismatch in {split_name}"
        )

    return len(images), total_instances, hashes


# ------------------------------------------------
# STEP 1: HASH EXISTING DATASET
# ------------------------------------------------

print("\n[1/7] Hashing existing images...")

existing_hashes = set()

for split in ["train", "valid", "val", "test"]:

    folder = EXISTING / split / "images"

    if folder.exists():
        for image in get_images(folder):
            existing_hashes.add(sha256(image))

print("Existing unique hashes:", len(existing_hashes))


# ------------------------------------------------
# STEP 2: READ KAGGLE SOURCE
# ------------------------------------------------

print("\n[2/7] Reading Kaggle source...")

kaggle_train = collect_source_images("train")
kaggle_val = collect_source_images("val")

print("Non-empty Kaggle train images:", len(kaggle_train))
print("Kaggle train instances:",
      sum(r["count"] for r in kaggle_train))

print("Non-empty Kaggle validation images:", len(kaggle_val))
print("Kaggle validation instances:",
      sum(r["count"] for r in kaggle_val))


# ------------------------------------------------
# STEP 3: REMOVE EXACT DUPLICATES
# ------------------------------------------------

print("\n[3/7] Checking duplicates...")

all_source = kaggle_train + kaggle_val

source_hashes = [r["hash"] for r in all_source]

if len(source_hashes) != len(set(source_hashes)):
    raise RuntimeError(
        "Duplicate images detected between Kaggle source splits. "
        "Resolve them before merging."
    )

filtered_train = [
    r for r in kaggle_train
    if r["hash"] not in existing_hashes
]

filtered_val = [
    r for r in kaggle_val
    if r["hash"] not in existing_hashes
]

print("Kaggle train images after duplicate removal:",
      len(filtered_train))

print("Kaggle validation images after duplicate removal:",
      len(filtered_val))


# ------------------------------------------------
# STEP 4: SELECT VALIDATION ADDITIONS
# ------------------------------------------------

print("\n[4/7] Selecting validation images...")

# First use all annotated Kaggle validation images.
validation_selected = list(filtered_val)

validation_count = sum(
    r["count"] for r in validation_selected
)

remaining_validation = VALID_TARGET - validation_count

print("Instances from Kaggle validation:", validation_count)
print("Additional instances needed:", max(0, remaining_validation))

# If needed, take additional images from Kaggle TRAIN.
# These images will NOT also be used in the output train split.
validation_from_train = []

if remaining_validation > 0:

    validation_from_train = select_closest(
        filtered_train,
        remaining_validation
    )

    validation_selected.extend(validation_from_train)

used_for_validation = {
    r["hash"] for r in validation_from_train
}

print("Validation images selected:", len(validation_selected))
print(
    "Validation instances selected:",
    sum(r["count"] for r in validation_selected)
)


# ------------------------------------------------
# STEP 5: SELECT TRAIN ADDITIONS
# ------------------------------------------------

print("\n[5/7] Selecting training images...")

remaining_train_pool = [
    r for r in filtered_train
    if r["hash"] not in used_for_validation
]

train_selected = select_closest(
    remaining_train_pool,
    TRAIN_TARGET
)

print("Training images selected:", len(train_selected))
print(
    "Training instances selected:",
    sum(r["count"] for r in train_selected)
)


# ------------------------------------------------
# STEP 6: CREATE OUTPUT DATASET
# ------------------------------------------------

print("\n[6/7] Creating updated dataset...")

for split in ["train", "valid", "test"]:

    (OUTPUT / split / "images").mkdir(parents=True)
    (OUTPUT / split / "labels").mkdir(parents=True)

# Copy existing dataset.
existing_train_count = copy_existing_split("train", OUTPUT / "train")

# Support either "valid" or "val" in existing dataset.
existing_valid_split = (
    "valid" if (EXISTING / "valid").exists() else "val"
)

existing_valid_count = copy_existing_split(
    existing_valid_split,
    OUTPUT / "valid"
)

existing_test_count = copy_existing_split(
    "test",
    OUTPUT / "test"
)

# Add selected Kaggle images.
copy_kaggle_records(train_selected, OUTPUT / "train")
copy_kaggle_records(validation_selected, OUTPUT / "valid")

print("Existing train images copied:", existing_train_count)
print("Existing validation images copied:", existing_valid_count)
print("Existing test images copied:", existing_test_count)


# ------------------------------------------------
# STEP 7: FINAL AUDIT
# ------------------------------------------------

print("\n[7/7] Running final audit...")

results = {}

for split in ["train", "valid", "test"]:

    images, instances, hashes = audit_output_split(split)

    results[split] = {
        "images": images,
        "instances": instances,
        "hashes": hashes
    }

# Check for cross-split image leakage.
all_hashes = []

for split in ["train", "valid", "test"]:
    all_hashes.extend(results[split]["hashes"])

if len(all_hashes) != len(set(all_hashes)):
    raise RuntimeError(
        "DUPLICATE IMAGE DETECTED ACROSS OUTPUT SPLITS!"
    )

print("\n" + "=" * 65)
print("FINAL UPDATED DOORS DATASET")
print("=" * 65)

print("\n{:<15} {:<15} {:<20}".format(
    "SPLIT", "IMAGES", "DOOR INSTANCES"
))

for split in ["train", "valid", "test"]:

    print("{:<15} {:<15} {:<20}".format(
        split.upper(),
        results[split]["images"],
        results[split]["instances"]
    ))

print("\nADDITIONS")

print("Train images added:", len(train_selected))
print(
    "Train instances added:",
    sum(r["count"] for r in train_selected)
)

print("Validation images added:", len(validation_selected))
print(
    "Validation instances added:",
    sum(r["count"] for r in validation_selected)
)

print("Test images added: 0")
print("Test instances added: 0")

print("\nDUPLICATE CHECK")
print("Cross-split exact duplicates: 0")
print("All output images have matching labels: YES")
print("All output door annotations use class ID 17: YES")

print("\nOUTPUT LOCATION:")
print(OUTPUT)

print("\n" + "=" * 65)
print("DATASET BUILD COMPLETE")
print("Original final_dataset was not modified.")
print("=" * 65)


BUILDING UPDATED DOORS DATASET

[1/7] Hashing existing images...
Existing unique hashes: 389

[2/7] Reading Kaggle source...
Non-empty Kaggle train images: 270
Kaggle train instances: 423
Non-empty Kaggle validation images: 35
Kaggle validation instances: 51

[3/7] Checking duplicates...
Kaggle train images after duplicate removal: 270
Kaggle validation images after duplicate removal: 35

[4/7] Selecting validation images...
Instances from Kaggle validation: 51
Additional instances needed: 19
Validation images selected: 48
Validation instances selected: 70

[5/7] Selecting training images...
Training images selected: 29
Training instances selected: 50

[6/7] Creating updated dataset...
Existing train images copied: 264
Existing validation images copied: 28
Existing test images copied: 97

[7/7] Running final audit...

FINAL UPDATED DOORS DATASET

SPLIT           IMAGES          DOOR INSTANCES      
TRAIN           293             320                 
VALID           76              100

In [8]:

# ================================================================
# DOORS DATASET - PERCEPTUAL DUPLICATE / LEAKAGE AUDIT
# ================================================================
# Scans updated_dataset only.
# Detects:
#   1. Exact duplicates (SHA-256)
#   2. Visually similar images (pHash)
#   3. Potential train-validation / train-test / valid-test leakage
#
# Does NOT delete or modify any images.
# ================================================================

!pip -q install ImageHash

import csv
import hashlib
from pathlib import Path
from collections import Counter
from PIL import Image
import imagehash

# ---------------- CONFIGURATION ----------------

DATASET = Path(
    "/content/drive/MyDrive/Hazard_Detection/raw_datasets/doors/updated_dataset"
)

REPORT_DIR = DATASET.parent / "duplicate_audit_reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Lower threshold = stricter matching.
# 0 = identical pHash, 4-6 = strong similarity candidates,
# 7-10 = broader similarity candidates.
PHASH_THRESHOLD = 6

SPLITS = ["train", "valid", "test"]

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}

# ------------------------------------------------
# VALIDATE DATASET
# ------------------------------------------------

if not DATASET.exists():
    raise FileNotFoundError(DATASET)

print("=" * 70)
print("DOORS PERCEPTUAL DUPLICATE AUDIT")
print("=" * 70)

# ------------------------------------------------
# HELPERS
# ------------------------------------------------

def sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


def hamming_distance(a, b):
    return (a ^ b).bit_count()


class BKTree:
    """BK-tree for efficient Hamming-distance searches."""

    def __init__(self):
        self.root = None

    def add(self, value, item_index):
        if self.root is None:
            self.root = [value, [item_index], {}]
            return

        node = self.root

        while True:
            distance = hamming_distance(value, node[0])

            if distance == 0:
                node[1].append(item_index)
                return

            if distance not in node[2]:
                node[2][distance] = [value, [item_index], {}]
                return

            node = node[2][distance]

    def search(self, value, threshold):
        matches = []

        if self.root is None:
            return matches

        stack = [self.root]

        while stack:
            node = stack.pop()

            distance = hamming_distance(value, node[0])

            if distance <= threshold:
                for index in node[1]:
                    matches.append((index, distance))

            low = distance - threshold
            high = distance + threshold

            for edge_distance, child in node[2].items():
                if low <= edge_distance <= high:
                    stack.append(child)

        return matches


# ------------------------------------------------
# STEP 1: COLLECT IMAGES AND HASHES
# ------------------------------------------------

records = []

print("\n[1/4] Reading images and calculating hashes...")

for split in SPLITS:

    image_dir = DATASET / split / "images"

    if not image_dir.exists():
        raise FileNotFoundError(image_dir)

    paths = sorted(
        p for p in image_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

    print(f"{split}: {len(paths)} images")

    for path in paths:

        try:
            exact_hash = sha256(path)

            with Image.open(path) as im:
                perceptual_hash = imagehash.phash(
                    im.convert("RGB")
                )

            records.append({
                "path": str(path),
                "relative_path": str(path.relative_to(DATASET)),
                "split": split,
                "sha256": exact_hash,
                "phash": int(str(perceptual_hash), 16)
            })

        except Exception as e:
            print("IMAGE ERROR:", path, "|", str(e))


print("Successfully processed:", len(records))


# ------------------------------------------------
# STEP 2: EXACT DUPLICATES
# ------------------------------------------------

print("\n[2/4] Checking exact duplicates...")

exact_groups = {}

for i, record in enumerate(records):
    exact_groups.setdefault(record["sha256"], []).append(i)

exact_pairs = []

for group in exact_groups.values():

    if len(group) < 2:
        continue

    for a in range(len(group)):
        for b in range(a + 1, len(group)):

            i, j = group[a], group[b]

            exact_pairs.append({
                "image_1": records[i]["relative_path"],
                "split_1": records[i]["split"],
                "image_2": records[j]["relative_path"],
                "split_2": records[j]["split"],
                "phash_distance": hamming_distance(
                    records[i]["phash"],
                    records[j]["phash"]
                ),
                "match_type": "EXACT_SHA256"
            })


# ------------------------------------------------
# STEP 3: PERCEPTUAL DUPLICATES
# ------------------------------------------------

print("\n[3/4] Checking perceptual similarities...")
print("pHash threshold:", PHASH_THRESHOLD)

tree = BKTree()
perceptual_pairs = []

for i, record in enumerate(records):

    matches = tree.search(
        record["phash"],
        PHASH_THRESHOLD
    )

    for j, distance in matches:

        # Exact duplicates are already reported separately.
        if records[i]["sha256"] == records[j]["sha256"]:
            continue

        perceptual_pairs.append({
            "image_1": records[j]["relative_path"],
            "split_1": records[j]["split"],
            "image_2": record["relative_path"],
            "split_2": record["split"],
            "phash_distance": distance,
            "match_type": "PERCEPTUAL_CANDIDATE"
        })

    tree.add(record["phash"], i)


# ------------------------------------------------
# STEP 4: SAVE REPORTS
# ------------------------------------------------

print("\n[4/4] Saving reports...")

all_pairs = exact_pairs + perceptual_pairs

csv_path = REPORT_DIR / "doors_duplicate_candidates.csv"

fieldnames = [
    "image_1",
    "split_1",
    "image_2",
    "split_2",
    "phash_distance",
    "match_type"
]

with open(csv_path, "w", newline="") as f:

    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_pairs)


# Cross-split leakage candidates.
cross_split = [
    pair for pair in all_pairs
    if pair["split_1"] != pair["split_2"]
]

cross_path = REPORT_DIR / "doors_cross_split_leakage.csv"

with open(cross_path, "w", newline="") as f:

    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(cross_split)


# Same-split duplicate candidates.
same_split = [
    pair for pair in all_pairs
    if pair["split_1"] == pair["split_2"]
]

same_path = REPORT_DIR / "doors_within_split_duplicates.csv"

with open(same_path, "w", newline="") as f:

    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(same_split)


# ------------------------------------------------
# SUMMARY
# ------------------------------------------------

print("\n" + "=" * 70)
print("PERCEPTUAL DUPLICATE AUDIT RESULTS")
print("=" * 70)

print("Images processed:", len(records))
print("Exact duplicate pairs:", len(exact_pairs))
print("Perceptual candidate pairs:", len(perceptual_pairs))
print("Cross-split candidate pairs:", len(cross_split))
print("Within-split candidate pairs:", len(same_split))

print("\nCross-split breakdown:")

for pair_type in [
    ("train", "valid"),
    ("train", "test"),
    ("valid", "test")
]:

    count = sum(
        1 for p in cross_split
        if {p["split_1"], p["split_2"]} == set(pair_type)
    )

    print(f"{pair_type[0]} <-> {pair_type[1]}: {count}")

print("\nREPORTS:")
print(csv_path)
print(cross_path)
print(same_path)

print("\nIMPORTANT:")
print("These are candidate matches, not automatic deletions.")
print("Review suspicious pairs visually before removing anything.")
print("=" * 70)


DOORS PERCEPTUAL DUPLICATE AUDIT

[1/4] Reading images and calculating hashes...
train: 293 images
valid: 76 images
test: 97 images
Successfully processed: 466

[2/4] Checking exact duplicates...

[3/4] Checking perceptual similarities...
pHash threshold: 6

[4/4] Saving reports...

PERCEPTUAL DUPLICATE AUDIT RESULTS
Images processed: 466
Exact duplicate pairs: 0
Perceptual candidate pairs: 3
Cross-split candidate pairs: 0
Within-split candidate pairs: 3

Cross-split breakdown:
train <-> valid: 0
train <-> test: 0
valid <-> test: 0

REPORTS:
/content/drive/MyDrive/Hazard_Detection/raw_datasets/doors/duplicate_audit_reports/doors_duplicate_candidates.csv
/content/drive/MyDrive/Hazard_Detection/raw_datasets/doors/duplicate_audit_reports/doors_cross_split_leakage.csv
/content/drive/MyDrive/Hazard_Detection/raw_datasets/doors/duplicate_audit_reports/doors_within_split_duplicates.csv

IMPORTANT:
These are candidate matches, not automatic deletions.
Review suspicious pairs visually before re